# 06 - Previsão de Palavras com BERTugues (Masked LM)

Neste notebook, vamos explorar a tarefa principal para a qual o BERT foi pré-treinado: **Masked Language Modeling (MLM)**.

O objetivo é repassar uma frase com uma lacuna (o token `[MASK]`) e deixar que o modelo utilize seu conhecimento da língua portuguesa para prever quais palavras fariam mais sentido naquele contexto.

Utilizaremos o modelo `ricardoz/BERTugues-base-portuguese-cased`.

## 1. Instalação e Importação
Precisamos apenas da biblioteca `transformers` e do `torch`.

In [1]:
# !pip install transformers torch

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM
import pandas as pd

## 2. Carregando o Pipeline de Preenchimento de Lacunas

A forma mais simples de testar o MLM é através da abstração `pipeline("fill-mask")` do Hugging Face. Ela já cuida da tokenização, predição e decodificação dos resultados automaticamente.

In [2]:
model_name = "ricardoz/BERTugues-base-portuguese-cased"

print("Carregando o modelo para Fill-Mask...")
unmasker = pipeline("fill-mask", model=model_name, device=0 if torch.cuda.is_available() else -1)

print("Modelo pronto para uso!")

Carregando o modelo para Fill-Mask...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo pronto para uso!


## 3. Exemplos de Previsão (Single Mask)

Vamos testar como o BERTugues completa frases comuns do nosso dia a dia.

In [3]:
frases = [
    "O Brasil é um [MASK] maravilhoso.",
    "Hoje o dia está com muito [MASK].",
    "Eu gosto de comer arroz com [MASK].",
    "O sol nasce no [MASK].",
    "Minha [MASK] favorita é a seleção brasileira."
]

for frase in frases:
    print(f"\nFRASE ORIGINAL: {frase}")
    resultados = unmasker(frase, top_k=3) # Pegar as 3 melhores predições
    
    for i, res in enumerate(resultados):
        print(f"  {i+1}º: {res['sequence']} (Score: {res['score']:.4f})")


FRASE ORIGINAL: O Brasil é um [MASK] maravilhoso.
  1º: O Brasil é um país maravilhoso. (Score: 0.7847)
  2º: O Brasil é um lugar maravilhoso. (Score: 0.0883)
  3º: O Brasil é um País maravilhoso. (Score: 0.0302)

FRASE ORIGINAL: Hoje o dia está com muito [MASK].
  1º: Hoje o dia está com muito movimento. (Score: 0.2018)
  2º: Hoje o dia está com muito sol. (Score: 0.1675)
  3º: Hoje o dia está com muito brilho. (Score: 0.0807)

FRASE ORIGINAL: Eu gosto de comer arroz com [MASK].
  1º: Eu gosto de comer arroz com feijão. (Score: 0.8898)
  2º: Eu gosto de comer arroz com arroz. (Score: 0.0288)
  3º: Eu gosto de comer arroz com batata. (Score: 0.0223)

FRASE ORIGINAL: O sol nasce no [MASK].
  1º: O sol nasce no outono. (Score: 0.0934)
  2º: O sol nasce no inverno. (Score: 0.0869)
  3º: O sol nasce no mar. (Score: 0.0729)

FRASE ORIGINAL: Minha [MASK] favorita é a seleção brasileira.
  1º: Minha equipe favorita é a seleção brasileira. (Score: 0.5441)
  2º: Minha seleção favorita é a sele

## 4. Análise de Contexto e Viés

O BERT é contextual. Se mudarmos as palavras ao redor do mask, a sugestão muda drasticamente.

In [4]:
exemplo_contexto = [
    "O jogador chutou a [MASK].",
    "O cozinheiro cortou a [MASK].",
    "O programador escreveu o [MASK]."
]

for f in exemplo_contexto:
    melhor_opcao = unmasker(f, top_k=1)[0]
    print(f"{f} -> {melhor_opcao['token_str']} ({melhor_opcao['score']:.4f})")

O jogador chutou a [MASK]. -> gol (0.4061)
O cozinheiro cortou a [MASK]. -> carne (0.2508)
O programador escreveu o [MASK]. -> texto (0.3318)


## 5. Mergulhando nos Tensores (Lógica Manual)

Para entender o que acontece "debaixo do capô", vamos fazer a mesma predição sem o pipeline, usando apenas o Tokenizador e o Modelo.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

text = "Gosto de ler [MASK] antes de dormir."

# 1. Tokenização
inputs = tokenizer(text, return_tensors="pt")
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# 2. Inferência
with torch.no_grad():
    logits = model(**inputs).logits

# 3. Pegar os Logits do token MASK e aplicar Softmax
mask_token_logits = logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

print(f"\nFrase: {text}")
print("Top 5 tokens manuais:")
for token_id in top_5_tokens:
    print(f" - {tokenizer.decode([token_id])}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Frase: Gosto de ler [MASK] antes de dormir.
Top 5 tokens manuais:
 - livros
 - muito
 - histórias
 - mais
 - romances
